<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/v4/multimodal/siglip-loss/siglip-loss.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("siglip-loss", ...)`


# SigLIP: Sigmoid Loss for Language–Image Pre-training

**Difficulty**: 🟢 Easy

### Problem Statement

CLIP trains its image and text towers with a **softmax** contrastive loss (InfoNCE; see `contrastive-loss-clip` in v3). Each image must pick its own caption out of the $N$ captions in the batch, so every loss term needs a normaliser over a whole row (and column) of the $N \times N$ similarity matrix. **SigLIP** (Zhai et al., *Sigmoid Loss for Language Image Pre-Training*, ICCV 2023) replaces this with an independent **binary classification for every (image, text) pair**: *do image $i$ and text $j$ belong together?* Nothing is normalised across the batch, so the loss is a plain sum over pairs. That sum can be computed block by block across devices without ever gathering the full similarity matrix. The sigmoid loss also beats softmax at small batch sizes and matches it at large ones. SigLIP image encoders are the vision towers of many open VLMs (PaliGemma, LLaVA-OneVision, Idefics2, …).

For a batch of $N$ pairs with image embeddings $x_i$ and text embeddings $y_j$ (rows of two $(N, D)$ matrices):

$$\hat x_i = \frac{x_i}{\lVert x_i\rVert_2},\qquad \hat y_j = \frac{y_j}{\lVert y_j\rVert_2},\qquad \ell_{ij} = t\,\hat x_i^{\top}\hat y_j + b,\qquad t = e^{t'}$$

$$z_{ij} = \begin{cases}+1 & i = j \ \text{(matching pair)}\\ -1 & i \neq j\end{cases}\qquad\qquad \mathcal L = -\frac{1}{N}\sum_{i=1}^{N}\sum_{j=1}^{N}\log\sigma\!\left(z_{ij}\,\ell_{ij}\right)$$

The log-temperature $t'$ (`logit_scale`) and the bias $b$ (`logit_bias`) are learnable scalars, which the paper initialises to $t' = \log 10$ and $b = -10$. Pay attention to the normalisation: the double sum over all $N^2$ pairs is divided by the **batch size $N$** (the paper's Algorithm 1), **not** by $N^2$.

---

### Requirements

1. **`siglip_loss(img_emb, txt_emb, logit_scale, logit_bias) -> Tensor`**
   - `img_emb`, `txt_emb`: `(N, D)` float tensors, **not** normalised. Row `i` of each forms matching pair `i`.
   - `logit_scale`: 0-dim tensor $t'$, the **log**-temperature. The cosine similarities are multiplied by $e^{t'}$.
   - `logit_bias`: 0-dim tensor $b$, added to every logit.
   - Return $\mathcal L$ as a 0-dim tensor (sum over all pairs, divided by $N$).
   - Differentiable w.r.t. all four inputs, because both scalars are trained.
   - Numerically stable: the loss and its gradients stay finite even when $|\ell_{ij}|$ is in the hundreds.
2. **`SigLIPHead(init_scale=math.log(10), init_bias=-10.0)`**, an `nn.Module` with
   - two learnable scalar `nn.Parameter`s named **`logit_scale`** (initialised to `init_scale`) and **`logit_bias`** (initialised to `init_bias`);
   - `forward(img_emb, txt_emb)` returning `siglip_loss(img_emb, txt_emb, self.logit_scale, self.logit_bias)`.

---

### Constraints

- ✅ Vectorised: one matmul builds the `(N, N)` logit matrix. No Python loops.
- ✅ Use `F.logsigmoid` or an equivalent stable form (`-F.softplus(-x)`, or `F.binary_cross_entropy_with_logits` with the right targets and reduction).
- ❌ No `softmax` / `F.cross_entropy`. SigLIP has no normalisation over the batch.
- ❌ No `math.exp(logit_scale)`, `.item()` or `float(...)` on the learnable scalars. Any of these cuts them out of the autograd graph.

---

<details>
  <summary>💡 Hint</summary>

  - Treat the $N \times N$ matrix as $N^2$ independent logistic-regression examples. Which ones are positives, and how many negatives are there for each positive?
  - `torch.eye(N)` marks the diagonal. What simple expression turns it into $+1$ on the diagonal and $-1$ everywhere else?
  - What does `torch.log(torch.sigmoid(x))` give for `x = -200` in float32? Then look up `F.logsigmoid`.
  - `.mean()` over the matrix divides by $N^2$, which is not the paper's normalisation.
  - Why would the bias start at $-10$ instead of $0$? Think about what fraction of the $N^2$ pairs are positives.

</details>

---

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Synthetic paired embeddings: image i and text i share a latent "concept", as if they came
# out of an image tower and a text tower that are already roughly aligned. Rows are NOT
# normalised and have very different norms on purpose.
torch.manual_seed(0)
N, D = 8, 32
concept = torch.randn(N, D)
img_emb = (concept + 0.5 * torch.randn(N, D)) * (torch.rand(N, 1) * 4 + 0.5)
txt_emb = (concept + 0.5 * torch.randn(N, D)) * (torch.rand(N, 1) * 4 + 0.5)
print("img_emb", tuple(img_emb.shape), " txt_emb", tuple(txt_emb.shape))
print("row norms:", [round(v, 1) for v in img_emb.norm(dim=1).tolist()])

In [ ]:
def siglip_loss(img_emb: torch.Tensor, txt_emb: torch.Tensor,
                logit_scale: torch.Tensor, logit_bias: torch.Tensor) -> torch.Tensor:
    """
    SigLIP pairwise sigmoid loss (Zhai et al., 2023, Algorithm 1).

    Args:
        img_emb:     (N, D) image embeddings, not normalised. Row i matches txt_emb[i].
        txt_emb:     (N, D) text embeddings, not normalised.
        logit_scale: 0-dim tensor t' (log-temperature); the logits are scaled by exp(t').
        logit_bias:  0-dim tensor b, added to every logit.

    Returns:
        0-dim tensor: -(1/N) * sum_ij log sigmoid(z_ij * logits_ij), with z = +1 on the
        diagonal and -1 off it.
    """
    # TODO: L2-normalise both embedding sets along the feature dim
    # TODO: logits = exp(logit_scale) * cosine-similarity matrix + logit_bias   -> (N, N)
    # TODO: labels z: +1 on the diagonal, -1 elsewhere                          -> (N, N)
    # TODO: stable log-sigmoid of z * logits, summed over all pairs, divided by N
    raise NotImplementedError

In [ ]:
# Test 1: the same loss written as binary cross-entropy with targets 1 on the diagonal and
# 0 elsewhere: -log sigmoid(z * l) == BCE-with-logits(l, (z + 1) / 2).
scale, bias = torch.tensor(math.log(10.0)), torch.tensor(-10.0)
loss = siglip_loss(img_emb, txt_emb, scale, bias)
assert loss.dim() == 0, f"expected a scalar, got shape {tuple(loss.shape)}"
logits = F.normalize(img_emb, dim=-1) @ F.normalize(txt_emb, dim=-1).T * scale.exp() + bias
oracle = F.binary_cross_entropy_with_logits(logits, torch.eye(N), reduction="sum") / N
assert torch.allclose(loss, oracle, atol=1e-5), \
    f"got {loss.item():.5f}, expected {oracle.item():.5f} (sum over pairs / N)"

# Test 2: only directions matter, because the loss L2-normalises the embeddings.
rescaled = siglip_loss(img_emb * (torch.rand(N, 1) * 10 + 0.1), txt_emb * 0.01, scale, bias)
assert torch.allclose(loss, rescaled, atol=1e-5), "rescaling the embeddings changed the loss"

# Test 3: temperature 1000 puts |logits| in the hundreds; the loss must stay finite.
big = siglip_loss(img_emb, txt_emb, torch.tensor(math.log(1000.0)), torch.tensor(0.0))
assert torch.isfinite(big), "loss is not finite for |logits| ~ 1000 -- use F.logsigmoid"

# Test 4: gradients reach the two learnable scalars.
s = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(-2.0, requires_grad=True)
siglip_loss(img_emb, txt_emb, s, b).backward()
assert s.grad is not None and b.grad is not None, "logit_scale / logit_bias got no gradient"
print(f"loss at the paper's init (t=10, b=-10): {loss.item():.4f}")
print("siglip_loss tests passed")

In [ ]:
class SigLIPHead(nn.Module):
    """SigLIP's learnable log-temperature and bias, plus the loss that uses them."""

    def __init__(self, init_scale: float = math.log(10.0), init_bias: float = -10.0):
        super().__init__()
        # TODO: self.logit_scale and self.logit_bias as scalar nn.Parameters
        raise NotImplementedError

    def forward(self, img_emb: torch.Tensor, txt_emb: torch.Tensor) -> torch.Tensor:
        """img_emb, txt_emb: (N, D) -> scalar SigLIP loss with the current parameters."""
        # TODO
        raise NotImplementedError

In [ ]:
head = SigLIPHead()
assert isinstance(head.logit_scale, nn.Parameter) and isinstance(head.logit_bias, nn.Parameter)
assert abs(head.logit_scale.exp().item() - 10.0) < 1e-4, "temperature exp(logit_scale) should start at 10"
assert abs(head.logit_bias.item() + 10.0) < 1e-6, "bias should start at -10"
assert torch.allclose(head(img_emb, txt_emb),
                      siglip_loss(img_emb, txt_emb, head.logit_scale, head.logit_bias))

# A tiny SigLIP training run: two linear "towers" on top of fixed features, mini-batches of 16.
torch.manual_seed(1)
n_pairs, d_img, d_txt, d_emb = 64, 48, 40, 16
latent = torch.randn(n_pairs, 8)
images = latent @ torch.randn(8, d_img) + 0.3 * torch.randn(n_pairs, d_img)
texts = latent @ torch.randn(8, d_txt) + 0.3 * torch.randn(n_pairs, d_txt)
img_tower, txt_tower, head = nn.Linear(d_img, d_emb), nn.Linear(d_txt, d_emb), SigLIPHead()
opt = torch.optim.Adam([*img_tower.parameters(), *txt_tower.parameters(), *head.parameters()], lr=1e-2)
for step in range(200):
    idx = torch.randperm(n_pairs)[:16]
    loss = head(img_tower(images[idx]), txt_tower(texts[idx]))
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 100 == 0 or step == 199:
        print(f"step {step:3d}  loss {loss.item():.3f}  t {head.logit_scale.exp().item():5.1f}"
              f"  b {head.logit_bias.item():6.2f}")
with torch.no_grad():
    sim = F.normalize(img_tower(images), dim=-1) @ F.normalize(txt_tower(texts), dim=-1).T
top1 = (sim.argmax(dim=1) == torch.arange(n_pairs)).float().mean().item()
print(f"image -> text top-1 retrieval over all {n_pairs} pairs: {top1:.0%}")
assert top1 > 0.5, "training did not align the pairs"
print("SigLIPHead tests passed")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("siglip-loss", siglip_loss=siglip_loss, SigLIPHead=SigLIPHead)